In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=10000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=1000)

torch.Size([8000, 10]) torch.Size([8000, 1])
torch.Size([2000, 10]) torch.Size([2000, 1])


In [12]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()
opt = optim.SGD(model.parameters())

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.5696668028831482
epoch:  1, loss: 0.5496978163719177
epoch:  2, loss: 0.5304321646690369
epoch:  3, loss: 0.5118458867073059
epoch:  4, loss: 0.49391525983810425
epoch:  5, loss: 0.4766175448894501
epoch:  6, loss: 0.4599306583404541
epoch:  7, loss: 0.4438333511352539
epoch:  8, loss: 0.42830538749694824
epoch:  9, loss: 0.4133271276950836
epoch:  10, loss: 0.3988797962665558
epoch:  11, loss: 0.38494521379470825
epoch:  12, loss: 0.3715059161186218
epoch:  13, loss: 0.35854506492614746
epoch:  14, loss: 0.34604644775390625
epoch:  15, loss: 0.33399441838264465
epoch:  16, loss: 0.3223738670349121
epoch:  17, loss: 0.311170369386673
epoch:  18, loss: 0.3003696799278259
epoch:  19, loss: 0.28995832800865173
epoch:  20, loss: 0.2799231708049774
epoch:  21, loss: 0.2702515423297882
epoch:  22, loss: 0.26093119382858276
epoch:  23, loss: 0.2519502341747284
epoch:  24, loss: 0.24329720437526703
epoch:  25, loss: 0.23496104776859283
epoch:  26, loss: 0.22693102061748505
e

In [13]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -2796.15869140625
Test metrics:  R2 = -2774.079833984375


In [11]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()
opt = optim.Adam(model.parameters())

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.0533682256937027
epoch:  1, loss: 0.04837410897016525
epoch:  2, loss: 0.043779656291007996
epoch:  3, loss: 0.03913073614239693
epoch:  4, loss: 0.03429394215345383
epoch:  5, loss: 0.030447984114289284
epoch:  6, loss: 0.030074235051870346
epoch:  7, loss: 0.028993064537644386
epoch:  8, loss: 0.02838001772761345
epoch:  9, loss: 0.027481578290462494
epoch:  10, loss: 0.02631313167512417
epoch:  11, loss: 0.0248650461435318
epoch:  12, loss: 0.022924885153770447
epoch:  13, loss: 0.020419573411345482
epoch:  14, loss: 0.01743846759200096
epoch:  15, loss: 0.014675011858344078
epoch:  16, loss: 0.012897193431854248
epoch:  17, loss: 0.011750100180506706
epoch:  18, loss: 0.0109419459477067
epoch:  19, loss: 0.010366547852754593
epoch:  20, loss: 0.009931626729667187
epoch:  21, loss: 0.009587622247636318
epoch:  22, loss: 0.009305344894528389
epoch:  23, loss: 0.009056360460817814
epoch:  24, loss: 0.008825713768601418
epoch:  25, loss: 0.00861698854714632
epoch:  2

In [14]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -2796.15869140625
Test metrics:  R2 = -2774.079833984375
